In [1]:
import secrets
import os
import pathlib
import base64

In [2]:
def gerar_chave(tamanho: int) -> bytes:
    """Gera uma chave criptograficamente segura com tamanho bytes."""
    chave_gerada_aleatoriamente = secrets.token_bytes(tamanho)
    return chave_gerada_aleatoriamente

def xor_bytes(dados: bytes, chave: bytes) -> bytes:
    """Calcula o XOR entre sequencias de mesmo comprimento."""
    if(len(dados)==len(chave)):
        # resultado_xor = bytearray()
        resultado_xor = []
        for i in range(len(dados)):
            resultado_xor.append(dados[i]^chave[i]) 
        return resultado_xor
    else:
        raise ValueError("Erro! Tamanhos diferentes entre dados e chave!")


def cifrar(mensagem: bytes, chave: bytes) -> bytes:
    """Cifra mensagem usando OTP."""
    if(len(mensagem)==len(chave)):
        cifrado_calculado = xor_bytes(mensagem,chave)
        return cifrado_calculado
    else:
        raise ValueError("Erro! Tamanhos diferentes entre mensagem e chave!")

def decifrar(cifrado: bytes, chave: bytes) -> bytes:
    """Decifra um texto cifrado usando OTP."""
    if(len(cifrado)==len(chave)):
        mensagem_decifrada = xor_bytes(cifrado,chave)
        return mensagem_decifrada
    else:
        raise ValueError("Erro! Tamanhos diferentes entre cifrado e chave!")

In [ ]:
import os
import hashlib

# Nomes dos arquivos de trabalho
arquivo_entrada = "registros_rede_sinteticos.csv"
arquivo_chave = "chave.key"
arquivo_cifrado = "arquivo.cifrado"
arquivo_recuperado = "arquivo.recuperado"

# 1. Leitura do arquivo original com "rb" (Read Binary)
with open(arquivo_entrada, "rb") as f:
    dados_originais = f.read()

# 2. Gerar a chave (com o mesmo tamanho do arquivo lido)
chave = gerar_chave(len(dados_originais))

# 3. Gravar a chave em disco temporariamente com "wb" (Write Binary)
with open(arquivo_chave, "wb") as f:
    f.write(chave)

# 4. Cifrar os dados
dados_cifrados = cifrar(dados_originais, chave)

# 5. Gravar resultados cifrados com "wb"
with open(arquivo_cifrado, "wb") as f:
    f.write(dados_cifrados)

# 6. Decifrar os dados
dados_decifrados = decifrar(dados_cifrados, chave)

# 7. Gravar resultados recuperados com "wb"
with open(arquivo_recuperado, "wb") as f:
    f.write(dados_decifrados)


# ==========================================
# EXIBIÇÃO DE RESULTADOS E VALIDAÇÕES
# ==========================================

# Registro dos tamanhos exigido no roteiro
print("--- Registro de Tamanhos ---")
print(f"Original:   {len(dados_originais)} bytes")
print(f"Cifrado:    {len(dados_cifrados)} bytes")
print(f"Recuperado: {len(dados_decifrados)} bytes")

# Verificação Criptográfica com SHA-256
print("\n--- Verificação de Integridade (SHA-256) ---")

# Calculando o hash em cima dos dados que lemos lá no início
hash_original = hashlib.sha256(dados_originais).hexdigest()

# Lendo o arquivo recuperado direto do disco com "rb" para provar que salvou certo
with open(arquivo_recuperado, "rb") as f:
    dados_lidos_do_disco = f.read()

hash_recuperado = hashlib.sha256(dados_lidos_do_disco).hexdigest()

print(f"Hash Original:   {hash_original}")
print(f"Hash Recuperado: {hash_recuperado}")

if hash_original == hash_recuperado:
    print("\nSUCESSO: Os hashes coincidem! O arquivo foi cifrado e recuperado com perfeição.")
else:
    print("\nFALHA: Os hashes são diferentes. O arquivo foi corrompido.")

# Remoção da cópia local da chave conforme o aviso
if os.path.exists(arquivo_chave):
    os.remove(arquivo_chave)
    print("\nAviso: O arquivo 'chave.key' foi removido do diretório por segurança.")